# NCAA March Madness 2026 - Basic EDA

**Objective:** Understand the basic structure of the data - teams, seasons, and game results.

**Outputs:**
- Summary statistics saved to `processed/basic_stats.csv`
- Data quality report
- Basic visualizations

## Setup

In [ ]:
import sys
sys.path.append('/home/sagemaker-user/NCAA')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import (
    load_teams, load_seasons, load_regular_season_results,
    load_tourney_results, load_seeds, validate_data_coverage
)
from src.utils import check_data_quality, get_season_summary

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Coverage Overview

In [ ]:
# Validate men's data coverage
print("MEN'S DATA COVERAGE")
print("=" * 60)
men_coverage = validate_data_coverage('M')
for key, value in men_coverage.items():
    if isinstance(value, list):
        print(f"{key}: {value[0]} to {value[-1]} ({len(value)} seasons)")
    else:
        print(f"{key}: {value:,}")

In [ ]:
# Validate women's data coverage
print("\nWOMEN'S DATA COVERAGE")
print("=" * 60)
women_coverage = validate_data_coverage('W')
for key, value in women_coverage.items():
    if isinstance(value, list):
        print(f"{key}: {value[0]} to {value[-1]} ({len(value)} seasons)")
    else:
        print(f"{key}: {value:,}")

## 2. Teams Analysis

In [ ]:
# Load teams
teams_m = load_teams('M')
teams_w = load_teams('W')

print("Men's Teams Sample:")
display(teams_m.head(10))

print("\nWomen's Teams Sample:")
display(teams_w.head(10))

In [ ]:
# Check data quality
check_data_quality(teams_m, "Men's Teams")
check_data_quality(teams_w, "Women's Teams")

In [ ]:
# Active teams in 2026
active_teams_m = teams_m[teams_m['LastD1Season'] == 2026]
print(f"Active Men's D1 Teams in 2026: {len(active_teams_m)}")
print(f"\nSample of active teams:")
display(active_teams_m.sample(10))

## 3. Seasons Analysis

In [ ]:
# Load seasons
seasons_m = load_seasons('M')
seasons_w = load_seasons('W')

print("Men's Seasons Sample:")
display(seasons_m.tail(10))

print("\nWomen's Seasons Sample:")
display(seasons_w.tail(10))

In [ ]:
# Example: Understanding DayNum
print("Understanding the DayNum System:")
print("DayZero is the reference date. All game days are offsets from this.")
print("\nExample for 2025 season:")
season_2025 = seasons_m[seasons_m['Season'] == 2025].iloc[0]
print(f"  DayZero: {season_2025['DayZero']}")
print(f"  DayNum=0 corresponds to: {season_2025['DayZero']}")
print(f"  DayNum=132 (Selection Sunday): ~March 15")
print(f"  DayNum=154 (Championship): ~April 6")

## 4. Regular Season Games Analysis

In [ ]:
# Load regular season results
regular_m = load_regular_season_results('M')

print("Regular Season Games Sample:")
display(regular_m.head(10))

check_data_quality(regular_m, "Men's Regular Season Games")

In [ ]:
# Games per season
games_per_season = regular_m.groupby('Season').size()

plt.figure(figsize=(14, 5))
plt.plot(games_per_season.index, games_per_season.values, marker='o')
plt.xlabel('Season')
plt.ylabel('Number of Games')
plt.title('Regular Season Games per Year (Men)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average games per season: {games_per_season.mean():.0f}")
print(f"Min: {games_per_season.min()} (Season {games_per_season.idxmin()})")
print(f"Max: {games_per_season.max()} (Season {games_per_season.idxmax()})")

In [ ]:
# Score distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Winning scores
axes[0].hist(regular_m['WScore'], bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(regular_m['WScore'].mean(), color='red', linestyle='--', 
                label=f"Mean: {regular_m['WScore'].mean():.1f}")
axes[0].set_xlabel('Winning Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Winning Score Distribution')
axes[0].legend()

# Losing scores
axes[1].hist(regular_m['LScore'], bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[1].axvline(regular_m['LScore'].mean(), color='red', linestyle='--',
                label=f"Mean: {regular_m['LScore'].mean():.1f}")
axes[1].set_xlabel('Losing Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Losing Score Distribution')
axes[1].legend()

# Victory margin
margin = regular_m['WScore'] - regular_m['LScore']
axes[2].hist(margin, bins=50, alpha=0.7, edgecolor='black', color='green')
axes[2].axvline(margin.mean(), color='red', linestyle='--',
                label=f"Mean: {margin.mean():.1f}")
axes[2].set_xlabel('Victory Margin')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Victory Margin Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Close games (margin ≤ 3 points): {(margin <= 3).sum():,} ({100*(margin <= 3).mean():.1f}%)")
print(f"Blowouts (margin > 20 points): {(margin > 20).sum():,} ({100*(margin > 20).mean():.1f}%)")

In [ ]:
# Home court advantage
home_games = regular_m[regular_m['WLoc'] == 'H']
away_games = regular_m[regular_m['WLoc'] == 'A']
neutral_games = regular_m[regular_m['WLoc'] == 'N']

total_games = len(regular_m)
print("Game Location Distribution:")
print(f"  Home wins: {len(home_games):,} ({100*len(home_games)/total_games:.1f}%)")
print(f"  Away wins: {len(away_games):,} ({100*len(away_games)/total_games:.1f}%)")
print(f"  Neutral: {len(neutral_games):,} ({100*len(neutral_games)/total_games:.1f}%)")

# Home court advantage in margin
home_margin = home_games['WScore'] - home_games['LScore']
away_margin = away_games['WScore'] - away_games['LScore']

print(f"\nAverage victory margin:")
print(f"  Home team wins: {home_margin.mean():.2f} points")
print(f"  Away team wins: {away_margin.mean():.2f} points")

In [ ]:
# Overtime analysis
ot_games = regular_m[regular_m['NumOT'] > 0]
print(f"Overtime Games: {len(ot_games):,} ({100*len(ot_games)/len(regular_m):.2f}%)")
print(f"\nOvertime Distribution:")
print(ot_games['NumOT'].value_counts().sort_index())

## 5. Tournament Games Analysis

In [ ]:
# Load tournament results
tourney_m = load_tourney_results('M')

print("Tournament Games Sample:")
display(tourney_m.head(10))

check_data_quality(tourney_m, "Men's Tournament Games")

In [ ]:
# Compare regular season vs tournament
print("REGULAR SEASON vs TOURNAMENT COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<30} {'Regular':<15} {'Tournament':<15}")
print("-" * 60)

reg_margin = regular_m['WScore'] - regular_m['LScore']
tour_margin = tourney_m['WScore'] - tourney_m['LScore']

print(f"{'Avg Winning Score':<30} {regular_m['WScore'].mean():<15.2f} {tourney_m['WScore'].mean():<15.2f}")
print(f"{'Avg Losing Score':<30} {regular_m['LScore'].mean():<15.2f} {tourney_m['LScore'].mean():<15.2f}")
print(f"{'Avg Victory Margin':<30} {reg_margin.mean():<15.2f} {tour_margin.mean():<15.2f}")
print(f"{'Overtime %':<30} {100*(regular_m['NumOT']>0).mean():<15.2f} {100*(tourney_m['NumOT']>0).mean():<15.2f}")
print(f"{'Close games (≤3 pts) %':<30} {100*(reg_margin<=3).mean():<15.2f} {100*(tour_margin<=3).mean():<15.2f}")

## 6. Tournament Seeds Analysis

In [ ]:
# Load seeds
seeds_m = load_seeds('M')

print("Tournament Seeds Sample:")
display(seeds_m.head(20))

# Extract numeric seed
seeds_m['SeedNum'] = seeds_m['Seed'].str[1:3].astype(int)
seeds_m['Region'] = seeds_m['Seed'].str[0]

print("\nSeed Distribution:")
print(seeds_m['SeedNum'].value_counts().sort_index())

In [ ]:
# Teams per region
recent_season = seeds_m['Season'].max()
recent_seeds = seeds_m[seeds_m['Season'] == recent_season]

print(f"Most Recent Tournament ({recent_season}):")
print(f"Total teams: {len(recent_seeds)}")
print(f"\nTeams per region:")
print(recent_seeds['Region'].value_counts())

## 7. Save Summary Statistics

In [ ]:
# Create summary statistics DataFrame
summary_stats = []

for season in sorted(regular_m['Season'].unique()):
    stats = get_season_summary(regular_m, season)
    summary_stats.append(stats)

summary_df = pd.DataFrame(summary_stats)

# Save to processed/
output_path = '/home/sagemaker-user/NCAA/processed/basic_stats.csv'
summary_df.to_csv(output_path, index=False)
print(f"✓ Saved summary statistics to {output_path}")

display(summary_df.tail(10))

## 8. Key Takeaways

**Write your observations here after running the analysis:**

1. Data coverage:
   - 
   
2. Game patterns:
   - 
   
3. Home court advantage:
   - 
   
4. Tournament vs Regular season:
   - 
   
5. Next steps:
   - Move to `02_eda_tournaments.ipynb` for seed analysis
   - Investigate specific teams or seasons of interest